# CMSC 173 &middot; Machine Learning &mdash; Week 14 Lab
## Under the Hood of Modern AI: Convolution & Attention

CNNs, ChatGPT, DALL-E &mdash; the models behind the headlines are built from two small operations you
can code in a few lines: **convolution** (how CNNs see images) and **attention** (how transformers
read text). You'll build both from scratch to demystify them, then reflect on the **ethics** of
deploying them. This is the course's last lab &mdash; a peek behind the curtain.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib (from scratch).** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + a tiny image

We make a small greyscale 'image' &mdash; a bright square on a dark background &mdash; so we can watch image
operations do something visible.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

img = np.zeros((20, 20))
img[5:15, 5:15] = 1.0                                  # a white square in the middle
img += 0.05 * np.random.default_rng(173).normal(size=img.shape)   # a little grain

plt.figure(figsize=(4,4)); plt.imshow(img, cmap='gray'); plt.title('our 20x20 image'); plt.axis('off'); plt.show()

**Reading the code:** a 20&times;20 grid of numbers &mdash; 0 is black, 1 is white. Real images are the same,
just bigger and with three colour channels. To a computer, 'an image' is exactly this array.

---
## Part 1 &middot; Convolution: how a CNN sees

A **convolution** slides a small grid of weights (a **kernel**) over the image and, at each spot,
multiplies-and-sums the pixels underneath. Different kernels detect different things &mdash; edges, blurs.
A CNN is just this operation with kernels it **learns** instead of ones we pick.

In [ ]:
def convolve(img, kernel):
    kh, kw = kernel.shape
    pad = kh // 2
    padded = np.pad(img, pad)                          # (1) border so output stays same size
    out = np.zeros_like(img)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            patch = padded[i:i+kh, j:j+kw]             # (2) the kh x kw window under the kernel
            out[i, j] = np.sum(patch * kernel)         # (3) multiply-and-sum
    return out

edge = np.array([[-1,-1,-1], [-1, 8,-1], [-1,-1,-1]])  # an edge-detecting kernel
edges = convolve(img, edge)

fig, ax = plt.subplots(1, 2, figsize=(8,4))
ax[0].imshow(img, cmap='gray');   ax[0].set_title('input');  ax[0].axis('off')
ax[1].imshow(edges, cmap='gray'); ax[1].set_title('after edge kernel'); ax[1].axis('off')
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `np.pad` adds a border of zeros so the output is the same size as the input.
- **(2)** for each pixel we grab the little window of neighbours under the kernel.
- **(3)** multiply the window by the kernel and sum &mdash; one number per position.
The edge kernel gives ~0 on flat regions (neighbours cancel) but lights up at the **borders** of the
square. That's how a CNN's first layers find edges &mdash; and later layers combine edges into shapes,
then objects.

**Answer here:**

1. The output is bright only at the square's outline, dark elsewhere. Why does this kernel respond
   to edges but ignore flat areas (hint: what do the &minus;1s and the 8 add up to)?
   &rarr; *your answer*

---
## Part 2 &middot; Different kernels, different eyes

Swap the kernel, change what the network 'sees'. A CNN *learns* which kernels are useful for its
task &mdash; here we just try two by hand.

In [ ]:
blur    = np.ones((3,3)) / 9                           # average the neighbours -> smooth
sharpen = np.array([[0,-1,0], [-1,5,-1], [0,-1,0]])    # emphasise the centre -> crisp

fig, ax = plt.subplots(1, 3, figsize=(11,4))
for a, (name, k) in zip(ax, [('blur', blur), ('sharpen', sharpen), ('edge', edge)]):
    a.imshow(convolve(img, k), cmap='gray'); a.set_title(name); a.axis('off')
plt.tight_layout(); plt.show()

**Reading the code:** `blur` averages each pixel with its neighbours (softening the grain); `sharpen`
boosts the centre over its neighbours (crisper edges); `edge` we saw already. Same convolution
machinery, three different behaviours &mdash; just from changing nine numbers.

---
## Part 3 &middot; Attention: how a transformer reads

Transformers (ChatGPT, BERT) process words by letting each word **look at** the others and decide
which matter. For each pair of words we score how related they are, then `softmax` turns those
scores into weights that sum to 1. We'll compute this **attention matrix** for a tiny sentence.

In [ ]:
words = ['the', 'cat', 'sat', 'on', 'mat']
# hand-made 4-number vector per word (real models LEARN these); related words point similarly
vecs = np.array([
    [1, 0, 0, 0],   # the
    [0, 1, 1, 0],   # cat
    [0, 0, 1, 1],   # sat
    [1, 0, 0, 0],   # on
    [0, 1, 1, 0],   # mat  (similar to 'cat')
], dtype=float)

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

scores = vecs @ vecs.T / np.sqrt(vecs.shape[1])       # (1) how related is every pair?
attention = softmax(scores)                            # (2) turn scores into weights that sum to 1

plt.figure(figsize=(5.5,4.5))
plt.imshow(attention, cmap='viridis')
plt.xticks(range(5), words); plt.yticks(range(5), words)
plt.xlabel('attends to ->'); plt.ylabel('word'); plt.colorbar(label='attention weight')
plt.title('Attention: who looks at whom'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `vecs @ vecs.T` scores every word pair by similarity (dot product); dividing by &radic;dim keeps
  the numbers tame.
- **(2)** `softmax` turns each row of scores into weights between 0 and 1 that sum to 1 &mdash; each word's
  'budget' of attention spread over the others.
In the heatmap, 'cat' and 'mat' (which we gave similar vectors) attend strongly to each other. A
real transformer stacks many of these layers, with **learned** vectors &mdash; but this is the core move.

**Answer here:**

1. In the heatmap, which word does 'cat' attend to most besides itself? We built that in &mdash; how?
   &rarr; *your answer*

2. Each row of the attention matrix sums to 1. Why is that a sensible design (think: a fixed budget
   of attention)?
   &rarr; *your answer*

---
## Part 4 &middot; The other half of the job: ethics

You can now build these systems. The harder question is *should* you, and *how*. This part has no
code &mdash; it's the reflection every ML practitioner owes. Answer honestly and specifically.

Consider a model trained on Philippine data and deployed for real decisions (loan approval, hiring,
medical triage, content moderation).

**Answer here** (a sentence or two each):

1. **Bias.** A hiring model trained on a company's past hires learns the company's past *biases*.
   Give one concrete way this could unfairly disadvantage a group of Filipino applicants.
   &rarr; *your answer*

2. **Deepfakes / generative AI.** GANs and diffusion models can fabricate convincing images, audio,
   and video. Name one harm this enables in a Philippine election or news context.
   &rarr; *your answer*

3. **Privacy.** RA 10173 (the Data Privacy Act) governs personal data here. Name one thing a
   student ML project should do to respect it when scraping or using people's data.
   &rarr; *your answer*

4. **Your line.** Name one ML application you would personally refuse to build, and why.
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| What a convolution kernel does | - |
| Why CNNs LEARN kernels rather than us picking them | - |
| The attention 'who looks at whom' idea | - |
| Why attention rows sum to 1 | - |
| One ethical risk of deployed ML | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what two simple operations power most of modern AI?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; A vertical-edge kernel

The edge kernel found all borders. Build a kernel that highlights only **vertical** edges (a Sobel
kernel: `[[-1,0,1],[-2,0,2],[-1,0,1]]`) and convolve the image. What lights up now? Fill it in.

In [ ]:
# your code here: sobel = np.array([[-1,0,1],[-2,0,2],[-1,0,1]]); imshow(convolve(img, sobel))

print('That is the whole course, from a mean and variance in week 2 to attention in week 14.')
print('Thanks for building all of it by hand.')

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 14

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/14/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 14 submission page](https://portal.latarak.com/course/cmsc173/lab/14/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.